# Final submission: test frames to `submission.csv`, from the stored weights

The whole pipeline of our final submission, with no network retrained. It loads three files from `weights/`
in the repository (written by `final/export.py`): the Swin U-Net trained on all 707 training frames, the
post-processing settings, and the component filter's training table (the filter, a small gradient boosting
with a fixed seed, is fitted from it in a few seconds; a pickled scikit-learn model would load only in the
version that wrote it).

1. **Input**: each 2048 x 2048 test frame becomes two channels, the frame and its limb-darkening-corrected copy
   (`cnn_baseline.data.channels`).
2. **Network**: the Swin U-Net (Swin-T encoder, U-Net decoder, trained with a boundary-weighted BCE + Dice loss)
   predicts 512 px tiles blended with a Hann window, for each of the 8 flips and quarter turns of the frame;
   the probability map is their mean (`zoo.experiment.predict_views`).
3. **Candidates**: hysteresis threshold 0.6 / 0.45, components of at least 300 px.
4. **Join**: pieces end to end along their axes within 50 px, any pieces within 10 px.
5. **Filter**: gradient boosting on shape, darkness, limb distance and the network's confidence keeps a
   filament when its chance of matching an annotator is worth it for PQ (cutoff chosen out of fold).
6. **Outline**: each kept filament is reduced to its pixels with probability >= 0.6.
7. **Submission**: one COCO RLE per filament.

Out of fold on the 707 training frames this scores PQ 0.4357; the public leaderboard gave 0.37.
Runs on a T4 GPU in about 20 minutes; on a CPU it works but takes hours. With the submitted file attached
(Kaggle: `filament-zoo-final-sweep`), the last cell checks that the new file reproduces it.

> Generated by `submission_pipeline/notebook.py` from `solution/final/pipeline.py` and the modules it imports. Edit those, not this notebook: the `%%writefile` cells are copies.

## Modules

The code this notebook imports, written to disk first.

In [ ]:
import os

os.makedirs("cnn_baseline", exist_ok=True)
os.makedirs("dataset", exist_ok=True)
os.makedirs("error_analysis", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("postprocess", exist_ok=True)
os.makedirs("preprocess", exist_ok=True)
os.makedirs("scoring", exist_ok=True)
os.makedirs("zoo", exist_ok=True)

In [ ]:
%%writefile dataset/__init__.py
"""The competition's data as distributed: where it lives and how the annotations are stored."""

IMAGE_SIZE = 2048  # every GONG H-alpha frame in the competition is IMAGE_SIZE x IMAGE_SIZE
N_TEST_IMAGES = 180

In [ ]:
%%writefile dataset/annotations.py
"""MAGFiLO COCO-style annotations.

One physical image (`file_name`) can carry up to three independent annotation sets from different
annotators. Each set is its own entry in `images`, with an id like `050102-20111116063134Lh`, and the
hosts score predictions against every set separately. Anything that splits data must therefore group
by `file_name`, never by entry id.
"""

import json
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

from pycocotools import mask as mask_util

from dataset import IMAGE_SIZE


@dataclass
class Annotations:
    file_to_entries: dict[str, list[str]]  # file_name -> annotator-image ids
    entry_to_rles: dict[str, list[dict]]  # annotator-image id -> one pycocotools RLE per filament
    n_annotations: int
    n_multi_polygon: int  # filaments drawn as more than one polygon (0 in the 2026 training set)

    def rles_for_file(self, file_name: str) -> list[list[dict]]:
        """Ground truth of one physical image: a list of filament RLEs per annotator."""
        return [self.entry_to_rles.get(entry, []) for entry in self.file_to_entries.get(file_name, [])]


def load_annotations(json_path: str | Path) -> Annotations:
    with open(json_path) as f:
        coco = json.load(f)
    file_to_entries = defaultdict(list)
    for image in coco["images"]:
        file_to_entries[image["file_name"]].append(image["id"])
    entry_to_rles = defaultdict(list)
    n_multi_polygon = 0
    for ann in coco["annotations"]:
        polygons = [p for p in ann["segmentation"] if len(p) >= 6]  # a polygon needs three points
        if not polygons:
            continue
        n_multi_polygon += len(polygons) > 1
        rle = mask_util.merge(mask_util.frPyObjects(polygons, IMAGE_SIZE, IMAGE_SIZE))
        entry_to_rles[ann["image_id"]].append(rle)
    return Annotations(dict(file_to_entries), dict(entry_to_rles), len(coco["annotations"]), n_multi_polygon)

In [ ]:
%%writefile dataset/paths.py
"""Where the competition data, attached notebook outputs and the writable output directory live."""

import os
from pathlib import Path

DATASET_DIR_NAME = "MAGFiLO_1.0_Kaggle_2026"


def input_root() -> Path:
    """Kaggle mounts the competition data and any attached notebook outputs under /kaggle/input.

    FILAMENT_INPUT overrides it, so a notebook can be smoke-tested locally on a synthetic copy.
    """
    return Path(os.environ.get("FILAMENT_INPUT", "/kaggle/input"))


def find_data_root() -> Path:
    """Locate the dataset folder; Kaggle's mount point has changed between releases, so search for it.

    FILAMENT_DATA points straight at the folder and skips the search.
    """
    if os.environ.get("FILAMENT_DATA"):
        return Path(os.environ["FILAMENT_DATA"])
    hits = sorted(input_root().glob(f"**/{DATASET_DIR_NAME}"))
    if not hits:
        raise FileNotFoundError(f"{DATASET_DIR_NAME} not found under {input_root()}")
    return hits[0]


def find_input(pattern: str) -> Path:
    """First file under the input root matching a glob, e.g. a `unet.pt` saved by another notebook."""
    hits = sorted(input_root().glob(f"**/{pattern}"))
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {input_root()}")
    return hits[0]


def output_dir() -> Path:
    """Kaggle's writable directory, else FILAMENT_OUTPUT, else the current directory."""
    kaggle_working = Path("/kaggle/working")
    if kaggle_working.is_dir():
        return kaggle_working
    return Path(os.environ.get("FILAMENT_OUTPUT", "."))

In [ ]:
%%writefile error_analysis/__init__.py
"""Where a submission loses Panoptic Quality, instance by instance.

PQ = sum of IoU over matches / (TP + 0.5 FP + 0.5 FN), a match being IoU > 0.5. Only three things
can go wrong, so the analysis classifies every ground-truth filament and every prediction by how it
relates to the other side (`instances`), aggregates the tables into what-if bounds and breakdowns by
size, contrast, position, site and annotator agreement (`report`), draws the worst cases (`gallery`),
and measures how far apart the human annotators are from each other (`annotators`), which is the
ceiling any model is scored against.

The tables are plain CSVs, so they come back from Kaggle and can be sliced locally without the images.

The notebooks live here too, next to the modules they use, like an approach directory under solution/:
`annotator_agreement.py` and `cnn_errors.py` run with `uv run filament run error_analysis/<name>.py --figures`
and become `filament-error-analysis-<name>`; `python -m error_analysis <outputs dir>` reprints a report locally.
"""

In [ ]:
%%writefile error_analysis/instances.py
"""Per-instance tables: every ground-truth filament and every prediction of a frame, classified.

Ground truth and predictions are linked wherever they share a pixel. The shape of those links, plus
the IoU > 0.5 rule of the metric, gives each instance one status:

    ground truth                      prediction
    matched     IoU > 0.5             matched        IoU > 0.5 with some filament
    near_miss   one overlap, IoU <= 0.5, and that prediction touches nothing else
    fragmented  covered by >= 2 preds  fragment       one of several covering a filament
    merged      swallowed by a pred    merge          covers >= 2 filaments
                that covers others
    missed      no overlap             hallucination  no overlap

Scoring is per annotator-image, so one prediction yields one row per annotator of its frame.
"""

from dataclasses import dataclass

import numpy as np
import pandas as pd
from pycocotools import mask as mask_util

from preprocess.disk import REL_SCALE
from scoring.metric import IOU_THRESHOLD, InstanceCrop, instance_crops


@dataclass
class FrameContext:
    """What the features of a frame's instances are computed from."""

    name: str  # file name, e.g. 20140609195854Bh.jpeg
    flat: np.ndarray  # uint8 limb-darkening-corrected frame (preprocess.disk.flatten)
    centre: tuple[float, float]  # disk centre in pixels
    radius: float  # disk radius in pixels
    prob: np.ndarray | None = None  # uint8 probability * 255 predicted by the model, if any

    @property
    def site(self) -> str:
        return self.name[14:16]

    @property
    def year(self) -> int:
        return int(self.name[:4])


def features(crop: np.ndarray, x: int, y: int, ctx: FrameContext) -> dict:
    """Shape and photometric features of one instance given its bounding-box crop (bool) at (x, y)."""
    ys, xs = np.nonzero(crop)
    h, w = crop.shape
    cx, cy = float(xs.mean()) + x, float(ys.mean()) + y
    if len(xs) > 2:
        eigenvalues = np.linalg.eigvalsh(np.cov(np.stack([xs, ys]).astype(float)))
        elongation = float(np.sqrt(max(eigenvalues[1], 1e-9) / max(eigenvalues[0], 1e-9)))
    else:
        elongation = 1.0
    flat = ctx.flat[y : y + h, x : x + w][crop]
    out = {
        "area": len(xs),
        "cx": cx,
        "cy": cy,
        "elongation": elongation,  # ratio of the principal axes; 1 = round, 10 = thin and long
        "darkness": float(1 - flat.mean() / REL_SCALE),  # 0.12 = 12 % darker than the local background
        "limb": float(np.hypot(cx - ctx.centre[0], cy - ctx.centre[1]) / ctx.radius) if ctx.radius else np.nan,
    }
    if ctx.prob is not None:
        prob = ctx.prob[y : y + h, x : x + w][crop]
        out.update(mean_prob=float(prob.mean() / 255), max_prob=float(prob.max() / 255))
    return out


def overlaps(gt_crops: list[InstanceCrop], labels: np.ndarray, pred_areas: np.ndarray):
    """(intersection, iou) matrices of shape (n_gt, n_pred), from the label map of the predictions."""
    n_pred = len(pred_areas)
    intersection = np.zeros((len(gt_crops), n_pred), dtype=np.int64)
    for i, (x, y, crop) in enumerate(gt_crops):
        h, w = crop.shape
        intersection[i] = np.bincount(labels[y : y + h, x : x + w][crop], minlength=n_pred + 1)[1:]
    gt_areas = np.array([int(crop.sum()) for _, _, crop in gt_crops], dtype=np.int64)
    union = gt_areas[:, None] + np.asarray(pred_areas, dtype=np.int64)[None, :] - intersection
    iou = np.where(union > 0, intersection / np.maximum(union, 1), 0.0)
    return intersection, iou


def classify(iou: np.ndarray) -> tuple[list[str], list[str]]:
    """Status of each ground-truth row and each prediction column of an IoU matrix."""
    touching = iou > 0
    gt_links, pred_links = touching.sum(axis=1), touching.sum(axis=0)
    hit = iou > IOU_THRESHOLD  # at most one per row and per column: two IoUs > 0.5 cannot share a side
    gt_status = []
    for i in range(iou.shape[0]):
        if gt_links[i] == 0:
            gt_status.append("missed")
        elif hit[i].any():
            gt_status.append("matched")
        elif gt_links[i] >= 2:
            gt_status.append("fragmented")
        else:
            gt_status.append("merged" if pred_links[int(iou[i].argmax())] >= 2 else "near_miss")
    pred_status = []
    for j in range(iou.shape[1]):
        if pred_links[j] == 0:
            pred_status.append("hallucination")
        elif hit[:, j].any():
            pred_status.append("matched")
        elif pred_links[j] >= 2:
            pred_status.append("merge")
        else:
            pred_status.append("fragment" if gt_links[int(iou[:, j].argmax())] >= 2 else "near_miss")
    return gt_status, pred_status


def agreement(rles_by_annotator: list[list[dict]]) -> list[np.ndarray]:
    """For each annotator's filaments: how many OTHER annotators drew a matching one (IoU > 0.5)."""
    counts = []
    for a, rles_a in enumerate(rles_by_annotator):
        count = np.zeros(len(rles_a), dtype=int)
        for b, rles_b in enumerate(rles_by_annotator):
            if a != b and rles_a and rles_b:
                iou = mask_util.iou(rles_a, rles_b, [0] * len(rles_b))  # (n_a, n_b)
                count += iou.max(axis=1) > IOU_THRESHOLD
        counts.append(count)
    return counts


def analyze_frame(
    ctx: FrameContext,
    annotators: list[tuple[str, list[dict]]],
    labels: np.ndarray,
    pred_areas: np.ndarray,
    pred_boxes: np.ndarray,
) -> tuple[list[dict], list[dict]]:
    """Rows for every (annotator, filament) and every (annotator, prediction) of one frame.

    `annotators` pairs each annotator-image id with its filament RLEs; `labels`, `pred_areas` and
    `pred_boxes` are the predictions as `postprocess.instances` returns them.
    """
    n_annotators = len(annotators)
    agreements = agreement([rles for _, rles in annotators])
    pred_features = []
    for k, (x, y, w, h) in enumerate(pred_boxes, start=1):
        crop = labels[y : y + h, x : x + w] == k
        pred_features.append({"pred_id": k, **features(crop, int(x), int(y), ctx)})
    frame_info = {"frame": ctx.name, "site": ctx.site, "year": ctx.year, "n_annotators": n_annotators}

    gt_rows, pred_rows = [], []
    for a, (annotator, rles) in enumerate(annotators):
        crops = instance_crops(rles)
        intersection, iou = overlaps(crops, labels, pred_areas)
        gt_status, pred_status = classify(iou)
        for i, (x, y, crop) in enumerate(crops):
            best = int(iou[i].argmax()) if iou.shape[1] else -1
            gt_rows.append(
                {
                    **frame_info,
                    "annotator": annotator,
                    "gt_id": i,
                    **features(crop, x, y, ctx),
                    "agreement": int(agreements[a][i]),  # other annotators who drew this filament too
                    "status": gt_status[i],
                    "n_overlap": int((iou[i] > 0).sum()),
                    "best_iou": float(iou[i, best]) if best >= 0 else 0.0,
                    "best_pred": best + 1 if best >= 0 and iou[i, best] > 0 else 0,
                    "coverage": float(intersection[i].sum() / max(crop.sum(), 1)),  # of its pixels under any pred
                }
            )
        for j, feats in enumerate(pred_features):
            best = int(iou[:, j].argmax()) if iou.shape[0] else -1
            pred_rows.append(
                {
                    **frame_info,
                    "annotator": annotator,
                    **feats,
                    "status": pred_status[j],
                    "n_overlap": int((iou[:, j] > 0).sum()),
                    "best_iou": float(iou[best, j]) if best >= 0 else 0.0,
                    "best_gt": best if best >= 0 and iou[best, j] > 0 else -1,
                    "precision": float(intersection[:, j].sum() / max(feats["area"], 1)),  # of its pixels on any GT
                }
            )
    return gt_rows, pred_rows


FRAME_COLUMNS = ["frame", "site", "year", "n_annotators", "annotator"]
FEATURE_COLUMNS = ["area", "cx", "cy", "elongation", "darkness", "limb", "mean_prob", "max_prob"]
GT_COLUMNS = [
    *FRAME_COLUMNS,
    "gt_id",
    *FEATURE_COLUMNS,
    "agreement",
    "status",
    "n_overlap",
    "best_iou",
    "best_pred",
    "coverage",
]
PRED_COLUMNS = [*FRAME_COLUMNS, "pred_id", *FEATURE_COLUMNS, "status", "n_overlap", "best_iou", "best_gt", "precision"]


def tables(gt_rows: list[dict], pred_rows: list[dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """DataFrames with a fixed column set, so an empty side (no predictions at all) still has a `status`."""
    gt = pd.DataFrame(gt_rows) if gt_rows else pd.DataFrame(columns=GT_COLUMNS)
    pred = pd.DataFrame(pred_rows) if pred_rows else pd.DataFrame(columns=PRED_COLUMNS)
    return gt, pred

In [ ]:
%%writefile models/__init__.py
"""Model architectures, independent of any training recipe. Each approach under solution/ picks one."""

In [ ]:
%%writefile models/edgeattnet.py
"""EdgeAttNet: the plain U-Net with edge-guided self-attention at its bottleneck.

Re-implemented from the paper's description (arXiv 2509.02964, on MAGFiLO), not from the authors' code. A
learnable edge map of the input -- 3 x 3 convolutions initialised as Sobel filters -- is pooled to the
bottleneck resolution and added to the features from which the attention's queries and keys are made, not
to its values. Positions on the same faint edges (a filament's border, its barbs) then attend to each other,
while what is passed along stays the image features. Everything else is `models.unet.UNet`.
"""

import torch
from torch import nn
from torch.nn import functional as F

from models.unet import UNet


class EdgeMap(nn.Module):
    """(B, C, H, W) image -> (B, width, H, W) edge responses; starts as |Sobel x|, |Sobel y| per channel."""

    def __init__(self, in_channels: int = 2, width: int = 16):
        super().__init__()
        self.sobel = nn.Conv2d(in_channels, 2 * in_channels, 3, padding=1, groups=in_channels, bias=False)
        kernel = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]]) / 4
        with torch.no_grad():
            self.sobel.weight.copy_(torch.stack([kernel, kernel.T]).repeat(in_channels, 1, 1).unsqueeze(1))
        self.mix = nn.Sequential(nn.Conv2d(2 * in_channels, width, 1), nn.BatchNorm2d(width), nn.ReLU(inplace=True))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mix(self.sobel(x).abs())


class EdgeGuidedAttention(nn.Module):
    """Multi-head self-attention over the bottleneck positions, queries and keys guided by the edge map."""

    def __init__(self, channels: int, edge_width: int, heads: int = 8):
        super().__init__()
        self.heads = heads
        self.norm = nn.GroupNorm(1, channels)
        self.edge = nn.Conv2d(edge_width, channels, 1)
        self.qk = nn.Conv2d(channels, 2 * channels, 1)
        self.v = nn.Conv2d(channels, channels, 1)
        self.out = nn.Conv2d(channels, channels, 1)
        nn.init.zeros_(self.out.weight)  # starts as the identity: the plain U-Net's bottleneck
        nn.init.zeros_(self.out.bias)

    def forward(self, x: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        features = self.norm(x)
        guided = features + self.edge(F.adaptive_avg_pool2d(edges, (h, w)))
        q, k = self.qk(guided).chunk(2, dim=1)
        v = self.v(features)

        def split(t):  # (B, C, H, W) -> (B, heads, HW, C / heads)
            return t.reshape(b, self.heads, c // self.heads, h * w).transpose(-1, -2)

        attended = F.scaled_dot_product_attention(split(q), split(k), split(v))
        return x + self.out(attended.transpose(-1, -2).reshape(b, c, h, w))


class EdgeAttNet(UNet):
    def __init__(self, in_channels: int = 2, base: int = 32, depth: int = 4, edge_width: int = 16, heads: int = 8):
        super().__init__(in_channels=in_channels, base=base, depth=depth)
        self.edge_map = EdgeMap(in_channels, edge_width)
        self.attention = EdgeGuidedAttention(base * 2**depth, edge_width, heads)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        edges = self.edge_map(x)
        skips = []
        for level, encoder in enumerate(self.encoders):
            skips.append(encoder(x if level == 0 else self.pool(skips[-1])))
        y = self.attention(skips.pop(), edges)
        for decoder in self.decoders:
            skip = skips.pop()
            y = F.interpolate(y, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            y = decoder(torch.cat([y, skip], dim=1))
        return self.head(y)

In [ ]:
%%writefile models/pretrained_unet.py
"""ImageNet encoders with a common full-resolution U-Net decoder."""

import segmentation_models_pytorch as smp
import torch
from torch import nn

ENCODERS = {
    "efficientnet_b3": "tu-efficientnet_b3.ra2_in1k",
    "convnext_tiny": "tu-convnext_tiny.fb_in1k",
    # keeps a 1/4-resolution branch through every stage instead of only downsampling, for thin structures
    "hrnet_w18": "tu-hrnet_w18.ms_aug_in1k",
}


class PretrainedUNet(nn.Module):
    """Two grayscale-derived channels; timm adapts RGB stem weights to two channels.

    Both channels use the same scalar ImageNet normalization, so neither acquires
    an arbitrary RGB colour identity. These constants are part of the experiment.
    """

    def __init__(self, encoder: str, pretrained: bool = True):
        super().__init__()
        self.net = smp.Unet(
            encoder_name=ENCODERS[encoder],
            encoder_weights="imagenet" if pretrained else None,
            in_channels=2,
            classes=1,
            decoder_channels=(256, 128, 64, 32, 16),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net((x - 0.449) / 0.226)

    def pretrained_parameters(self):
        """The ImageNet-initialised part, trained with a smaller learning rate than the decoder."""
        return self.net.encoder.parameters()

In [ ]:
%%writefile models/segformer.py
"""SegFormer: a hierarchical transformer encoder (Mix Transformer) with its light all-MLP decoder.

The encoder's self-attention sees the whole tile, so a long filament is one context rather than a chain of
local patches. The decoder predicts at 1/4 of the input resolution and is upsampled, as in the paper; whether
that loses the 10-20 px wide filaments is part of what the experiment measures.
"""

import segmentation_models_pytorch as smp
import torch
from torch import nn


class SegFormer(nn.Module):
    """MiT-B2 encoder (ImageNet), two input channels (smp adapts the RGB patch embedding), logits out."""

    def __init__(self, variant: str = "mit_b2", pretrained: bool = True):
        super().__init__()
        self.net = smp.Segformer(
            encoder_name=variant, encoder_weights="imagenet" if pretrained else None, in_channels=2, classes=1
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net((x - 0.449) / 0.226)

    def pretrained_parameters(self):
        return self.net.encoder.parameters()

In [ ]:
%%writefile models/swin_unet.py
"""Swin-UNet: a Swin Transformer encoder with a U-Net decoder and skip connections.

Shifted-window self-attention gives each 1/4-resolution token a context of whole windows that move between
blocks, so a filament can be followed across a tile; the U-Net decoder brings the four stages (1/4 .. 1/32)
back up. Swin starts at 1/4 resolution, too coarse for 10-20 px wide filaments on its own, so two small
convolutional stages at full and half resolution supply the last two skip connections.

The encoder is timm's ImageNet Swin-T, rebuilt for 512 px tiles with 8 x 8 windows (512 / 4 / 8 = 16
windows per side at every stage); timm resizes the pretrained position tables.
"""

import timm
import torch
from torch import nn
from torch.nn import functional as F

from models.unet import conv_block

ENCODER = "swin_tiny_patch4_window7_224.ms_in22k_ft_in1k"


class SwinUNet(nn.Module):
    def __init__(self, pretrained: bool = True, tile: int = 512, stem: tuple[int, int] = (16, 32)):
        super().__init__()
        self.encoder = timm.create_model(
            ENCODER,
            pretrained=pretrained,
            features_only=True,
            in_chans=2,
            img_size=tile,
            window_size=8,
            strict_img_size=False,
        )
        widths = self.encoder.feature_info.channels()  # 96, 192, 384, 768 at 1/4 .. 1/32
        self.stem_full = conv_block(2, stem[0])
        self.stem_half = conv_block(stem[0], stem[1])
        skips = [stem[0], stem[1], *widths[:-1]]  # 1/1, 1/2, 1/4, 1/8, 1/16
        decoder = [stem[0], stem[1], *widths[:-1]]
        self.decoders = nn.ModuleList()
        previous = widths[-1]
        for skip, width in zip(reversed(skips), reversed(decoder)):
            self.decoders.append(conv_block(previous + skip, width))
            previous = width
        self.head = nn.Conv2d(stem[0], 1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = (x - 0.449) / 0.226
        full = self.stem_full(x)
        half = self.stem_half(F.max_pool2d(full, 2))
        stages = [f.permute(0, 3, 1, 2) for f in self.encoder(x)]  # timm's Swin features are NHWC
        skips = [full, half, *stages[:-1]]
        y = stages[-1]
        for decoder, skip in zip(self.decoders, reversed(skips)):
            y = F.interpolate(y, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            y = decoder(torch.cat([y, skip], dim=1))
        return self.head(y)

    def pretrained_parameters(self):
        return self.encoder.parameters()

In [ ]:
%%writefile models/unet.py
"""A plain U-Net: the ordinary convolutional encoder-decoder for segmentation, written from scratch.

No pretrained backbone, no attention. Each level is two 3x3 conv-BN-ReLU; the encoder halves the
resolution with max-pooling, the decoder doubles it bilinearly and concatenates the skip connection.
It is fully convolutional, so it can train on tiles and predict whole 2048 x 2048 frames.
"""

import torch
from torch import nn
from torch.nn import functional as F


def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """`depth` down-sampling steps; channel widths base, 2*base, ... at each level. Output: logits."""

    def __init__(self, in_channels: int = 2, out_channels: int = 1, base: int = 32, depth: int = 4):
        super().__init__()
        widths = [base * 2**level for level in range(depth + 1)]
        self.encoders = nn.ModuleList()
        previous = in_channels
        for width in widths:
            self.encoders.append(conv_block(previous, width))
            previous = width
        self.decoders = nn.ModuleList(
            conv_block(widths[level + 1] + widths[level], widths[level]) for level in reversed(range(depth))
        )
        self.pool = nn.MaxPool2d(2)
        self.head = nn.Conv2d(base, out_channels, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for level, encoder in enumerate(self.encoders):
            x = encoder(x if level == 0 else self.pool(x))
            skips.append(x)
        x = skips.pop()
        for decoder in self.decoders:
            skip = skips.pop()
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = decoder(torch.cat([x, skip], dim=1))
        return self.head(x)

In [ ]:
%%writefile postprocess/__init__.py
"""Post-processing shared by the approaches: probability map -> joined, filtered filament instances."""

In [ ]:
%%writefile postprocess/filtering.py
"""A second stage that decides, per predicted component, whether it is a filament.

The minimum-area cut is a crude false-positive filter: at 600 px it removes a fifth of the real
filaments to get rid of hallucinations. A small classifier on the component's shape, contrast,
position and the model's confidence separates them better and lets the area cut drop.

It is trained on the validation frames, the only frames whose predictions are honest, and its
cutoff is chosen on out-of-fold scores so the validation PQ it reports is not fitted to itself.
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GroupKFold

from error_analysis.instances import FrameContext, features, overlaps
from preprocess.disk import disk_geometry, flatten
from scoring.metric import IOU_THRESHOLD, InstanceCrop

FEATURES = ["log_area", "elongation", "darkness", "limb", "mean_prob", "max_prob"]
MIN_TRAINING_COMPONENTS = 60  # below this (or with one class only) the filter is not fitted


def frame_context(name: str, image: np.ndarray, prob: np.ndarray) -> FrameContext:
    cx, cy, radius = disk_geometry(image)
    return FrameContext(name=name, flat=flatten(image), centre=(cx, cy), radius=radius, prob=prob)


def component_table(ctx: FrameContext, labels: np.ndarray, boxes: np.ndarray) -> pd.DataFrame:
    """One row per predicted component with the FEATURES columns (and frame / component id)."""
    rows = []
    for k, (x, y, w, h) in enumerate(boxes, start=1):
        crop = labels[y : y + h, x : x + w] == k
        rows.append({"frame": ctx.name, "component": k, **features(crop, int(x), int(y), ctx)})
    table = pd.DataFrame(
        rows,
        columns=["frame", "component", "area", "cx", "cy", "elongation", "darkness", "limb", "mean_prob", "max_prob"],
    )
    table["log_area"] = np.log(table["area"].astype(float).clip(lower=1))
    return table


def matched_by_any_annotator(
    gt_crops_per_annotator: list[list[InstanceCrop]], labels: np.ndarray, areas: np.ndarray
) -> np.ndarray:
    """bool per component: IoU > 0.5 with a filament of at least one annotator.

    Such a component raises PQ whenever it is kept (its IoU outweighs the annotators it misses); a
    component matching nobody can only lower it. That makes this the right target for the filter.
    """
    keep = np.zeros(len(areas), dtype=bool)
    for crops in gt_crops_per_annotator:
        if crops and len(areas):
            _, iou = overlaps(crops, labels, areas)
            keep |= (iou > IOU_THRESHOLD).any(axis=0)
    return keep


class ComponentFilter:
    """Gradient boosting on `features`; `cutoff` is the probability below which a component is dropped."""

    def __init__(self, cutoff: float = 0.5, features: list[str] = FEATURES):
        self.cutoff, self.features = cutoff, list(features)
        self.model = GradientBoostingClassifier(n_estimators=150, max_depth=2, learning_rate=0.05, random_state=0)

    def fit(self, table: pd.DataFrame, target: np.ndarray) -> "ComponentFilter":
        self.model.fit(table[self.features].to_numpy(), np.asarray(target, dtype=int))
        return self

    def scores(self, table: pd.DataFrame) -> np.ndarray:
        if not len(table):
            return np.zeros(0)
        return self.model.predict_proba(table[self.features].to_numpy())[:, 1]

    def keep(self, table: pd.DataFrame) -> np.ndarray:
        return self.scores(table) >= self.cutoff


def out_of_fold_scores(
    table: pd.DataFrame, target: np.ndarray, n_splits: int = 5, features: list[str] = FEATURES
) -> np.ndarray:
    """Filter probabilities for every component, each predicted by a model that never saw its frame."""
    scores = np.zeros(len(table))
    groups = table["frame"].to_numpy()
    splits = min(n_splits, len(np.unique(groups)))
    for train_idx, test_idx in GroupKFold(n_splits=splits).split(table, target, groups):
        model = ComponentFilter(features=features).fit(table.iloc[train_idx], target[train_idx])
        scores[test_idx] = model.scores(table.iloc[test_idx])
    return scores


def trainable(target: np.ndarray) -> bool:
    return len(target) >= MIN_TRAINING_COMPONENTS and 0 < target.sum() < len(target)

In [ ]:
%%writefile postprocess/instances.py
"""Probability map -> filament instances: (hysteresis) threshold and connected components; label-map bookkeeping."""

import cv2
import numpy as np

from preprocess.disk import disk_mask

MAX_INSTANCES = 40  # ground truth never exceeds 26 filaments per frame
DISK_SLACK = 1.02  # keep a sliver past the estimated limb so filaments touching it are not clipped


def on_disk(prob: np.ndarray, image: np.ndarray) -> np.ndarray:
    """`prob` with everything off the solar disk set to 0.

    Training tiles are centred on the disk, so the network barely sees the sky beyond the limb and its
    output there hovers around 0.5 -- enough to pass a 0.5 threshold as large spurious blobs.
    """
    return prob * disk_mask(image, shrink=DISK_SLACK)


def instances(prob: np.ndarray, threshold: float, min_area: int, low: float | None = None):
    """(label map with 0 = background and 1..K, areas[K], boxes[K, 4] as x, y, w, h), largest first.

    `prob` is uint8 (H, W) holding probability * 255, as `training.predict` yields it.

    With `low` (< threshold) the threshold is hysteretic: a component of `prob >= low` is kept whole
    when any of its pixels reaches `threshold`. Filaments fade towards their ends, so a single
    threshold keeps the confident core and drops the faint tails; hysteresis follows them.
    """
    high = prob >= round(threshold * 255)
    if low is not None and low < threshold:
        n_low, low_labels = cv2.connectedComponents((prob >= round(low * 255)).astype(np.uint8), connectivity=8)
        seeded = np.zeros(n_low, dtype=bool)
        seeded[np.unique(low_labels[high])] = True
        seeded[0] = False
        foreground = seeded[low_labels].astype(np.uint8)
    else:
        foreground = high.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(foreground, connectivity=8)
    areas = stats[:, cv2.CC_STAT_AREA]
    keep = np.flatnonzero(areas >= min_area)
    keep = keep[keep != 0]  # component 0 is the background
    keep = keep[np.argsort(areas[keep])[::-1][:MAX_INSTANCES]]
    relabel = np.zeros(n, dtype=np.int32)
    relabel[keep] = np.arange(1, len(keep) + 1)
    return relabel[labels], areas[keep], stats[keep, :4]


def select(labels: np.ndarray, areas: np.ndarray, boxes: np.ndarray, keep: np.ndarray):
    """Drop the instances where `keep` (bool per instance 1..K) is False; renumber the rest 1..K'."""
    keep = np.asarray(keep, dtype=bool)
    relabel = np.zeros(len(areas) + 1, dtype=np.int32)
    relabel[1:][keep] = np.arange(1, int(keep.sum()) + 1)
    return relabel[labels], areas[keep], boxes[keep]


def cut(labels: np.ndarray, areas: np.ndarray, boxes: np.ndarray, prob: np.ndarray, threshold: float):
    """The instances reduced to their pixels with `prob` >= threshold, renumbered; ones left under 30 px dropped.

    Hysteresis decides which pixels belong together and which filaments exist; this decides their outline.
    The network's 0.3 contour is wider than the annotators' (out of fold, cutting at 0.6 raised PQ by 0.006).
    """
    if not len(areas):
        return labels, areas, boxes
    labels = np.where(prob >= round(threshold * 255), labels, 0)
    areas, boxes = label_stats(labels, len(areas))
    return select(labels, areas, boxes, areas >= 30)


def label_stats(labels: np.ndarray, n: int) -> tuple[np.ndarray, np.ndarray]:
    """(areas[n], boxes[n, 4] as x, y, w, h) of the instances 1..n of a label map (empty ones: area 0)."""
    ys, xs = np.nonzero(labels)
    ids = labels[ys, xs] - 1
    areas = np.bincount(ids, minlength=n)[:n]
    x0, y0 = np.full(n, labels.shape[1]), np.full(n, labels.shape[0])
    x1, y1 = np.full(n, -1), np.full(n, -1)
    np.minimum.at(x0, ids, xs)
    np.minimum.at(y0, ids, ys)
    np.maximum.at(x1, ids, xs)
    np.maximum.at(y1, ids, ys)
    boxes = np.stack([x0, y0, x1 - x0 + 1, y1 - y0 + 1], axis=1)
    boxes[areas == 0] = 0
    return areas, boxes


def renumber(labels: np.ndarray, mapping: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Apply `mapping` (old label -> group, 0 stays 0), then number the groups 1..K largest first."""
    grouped = mapping[labels]
    groups = np.unique(mapping[1:])
    groups = groups[groups > 0]
    compact = np.zeros(int(mapping.max()) + 1, dtype=np.int32)
    compact[groups] = np.arange(1, len(groups) + 1)
    grouped = compact[grouped]
    areas, _ = label_stats(grouped, len(groups))
    order = np.argsort(areas, kind="stable")[::-1]
    by_size = np.zeros(len(groups) + 1, dtype=np.int32)
    by_size[order + 1] = np.arange(1, len(groups) + 1)
    out = by_size[grouped]
    areas, boxes = label_stats(out, len(groups))
    return out, areas, boxes


def merge(labels: np.ndarray, links) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Merge the instances of each (a, b) in `links`, transitively; renumber largest first. No pixels are added."""
    n = int(labels.max())
    parent = np.arange(n + 1)

    def root(k):
        while parent[k] != k:
            parent[k] = parent[parent[k]]
            k = parent[k]
        return k

    for a, b in links:
        parent[root(int(b))] = root(int(a))
    return renumber(labels, np.array([root(k) for k in range(n + 1)]))

In [ ]:
%%writefile postprocess/joining.py
"""Which kept components belong to the same filament: features of nearby pairs, and joining by a decision.

Joining kept components within a fixed distance helped (`variants.join`), but distance alone cannot tell
how two pieces relate. Physically a filament breaks in two ways:

- **along its spine**, where the material thins out: the pieces line up end to end, and the gap runs
  along their common axis, often still a little darker than the surroundings;
- **at a barb**, a short side extension from the spine down to the chromosphere: a small piece next to
  the *side* of the spine, at an angle to it.

Two separate filaments, in contrast, lie side by side or end to end in different filament channels. The
features below describe a pair in those terms: distance, the angles between the axes and the gap, whether
the gap leaves the larger piece at its end, how dark and how confident the gap is, and the direction of
the chromospheric fibrils around the gap (fibrils in a filament channel run along the channel).
"""

import cv2
import numpy as np
import pandas as pd
from scipy import ndimage

from error_analysis.instances import FrameContext, overlaps
from postprocess.instances import label_stats, merge, select
from preprocess.disk import REL_SCALE
from scoring.metric import InstanceCrop

PAIR_FEATURES = [
    "dist",
    "angle_diff",
    "gap_large",
    "gap_small",
    "end_large",
    "end_small",
    "path_dark",
    "path_prob_mean",
    "path_prob_min",
    "fibril_angle",
    "fibril_coherence",
    "log_area_large",
    "log_area_small",
    "elong_large",
    "elong_small",
    "limb",
]
ORIENTED = 2.0  # elongation above which a piece has a meaningful axis
FIBRIL_HALF = 20  # half-width (px) of the window around the gap in which the fibril direction is measured


def line_angle(a, b):
    """Angle between two undirected lines given by their angles (radians), in [0, pi / 2]."""
    d = np.abs(np.asarray(a) - np.asarray(b)) % np.pi
    return np.minimum(d, np.pi - d)


def component_axes(labels: np.ndarray, n: int) -> pd.DataFrame:
    """Per instance 1..n: centroid, principal-axis direction (unit vector, angle in [0, pi)), elongation and
    the extent along the axis (lo..hi, relative to the centroid)."""
    _, boxes = label_stats(labels, n)
    rows = []
    for k, (x, y, w, h) in enumerate(boxes, start=1):
        ys, xs = np.nonzero(labels[y : y + h, x : x + w] == k)
        xs, ys = (xs + x).astype(float), (ys + y).astype(float)
        cx, cy = xs.mean(), ys.mean()
        if len(xs) > 2:
            values, vectors = np.linalg.eigh(np.cov(np.stack([xs, ys])))
            ux, uy = vectors[:, 1]
            elongation = float(np.sqrt(max(values[1], 1e-9) / max(values[0], 1e-9)))
        else:
            ux, uy, elongation = 1.0, 0.0, 1.0
        proj = (xs - cx) * ux + (ys - cy) * uy
        rows.append(
            {
                "cx": cx,
                "cy": cy,
                "ux": ux,
                "uy": uy,
                "angle": float(np.arctan2(uy, ux) % np.pi),
                "elongation": elongation,
                "lo": proj.min(),
                "hi": proj.max(),
                "area": len(xs),
            }
        )
    return pd.DataFrame(rows, columns=["cx", "cy", "ux", "uy", "angle", "elongation", "lo", "hi", "area"])


def pairs(labels: np.ndarray, radius: int) -> pd.DataFrame:
    """Every pair a < b of instances within `radius` px: their distance and nearest pixels (ax, ay), (bx, by)."""
    n = int(labels.max())
    _, boxes = label_stats(labels, n)
    rows = []
    for a, (x, y, w, h) in enumerate(boxes, start=1):
        x0, y0 = max(x - radius, 0), max(y - radius, 0)
        x1, y1 = min(x + w + radius, labels.shape[1]), min(y + h + radius, labels.shape[0])
        window = labels[y0:y1, x0:x1]
        dist, (iy, ix) = ndimage.distance_transform_edt(window != a, return_indices=True)
        for b in np.unique(window[(dist <= radius) & (window > a)]):
            d = np.where(window == b, dist, np.inf)
            by, bx = np.unravel_index(int(np.argmin(d)), d.shape)
            rows.append(
                {
                    "a": a,
                    "b": int(b),
                    "dist": float(d[by, bx]),
                    "ax": int(ix[by, bx]) + x0,
                    "ay": int(iy[by, bx]) + y0,
                    "bx": int(bx) + x0,
                    "by": int(by) + y0,
                }
            )
    return pd.DataFrame(rows, columns=["a", "b", "dist", "ax", "ay", "bx", "by"])


def path_values(image: np.ndarray, p: tuple[int, int], q: tuple[int, int]) -> np.ndarray:
    """Values of `image` on the straight path from pixel p to pixel q (x, y), endpoints excluded when possible."""
    m = max(int(np.hypot(q[0] - p[0], q[1] - p[1])) - 1, 1)
    t = np.linspace(0, 1, m + 2)[1:-1]
    xs = np.rint(p[0] + t * (q[0] - p[0])).astype(int)
    ys = np.rint(p[1] + t * (q[1] - p[1])).astype(int)
    return image[ys, xs]


def fibril_direction(image: np.ndarray, x: int, y: int, exclude: np.ndarray, half: int = FIBRIL_HALF):
    """(angle of the dominant line direction in [0, pi), coherence in [0, 1]) around (x, y).

    Structure tensor of the image gradients: the dominant gradient direction is across the fibrils, so the
    fibrils run perpendicular to it. Pixels where `exclude` is set (the components themselves, whose edges
    would otherwise dominate) do not count. Coherence 0 = no preferred direction, 1 = perfectly parallel lines.
    """
    x0, y0 = max(x - half, 0), max(y - half, 0)
    x1, y1 = min(x + half + 1, image.shape[1]), min(y + half + 1, image.shape[0])
    window = image[y0:y1, x0:x1].astype(np.float32)
    gx = cv2.Sobel(window, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(window, cv2.CV_32F, 0, 1, ksize=3)
    weight = (~exclude[y0:y1, x0:x1]).astype(np.float32)
    jxx, jyy, jxy = (float((weight * g).sum()) for g in (gx * gx, gy * gy, gx * gy))
    if jxx + jyy <= 0:
        return 0.0, 0.0
    gradient = 0.5 * np.arctan2(2 * jxy, jxx - jyy)
    coherence = np.hypot(jxx - jyy, 2 * jxy) / (jxx + jyy)
    return float((gradient + np.pi / 2) % np.pi), float(coherence)


def end_position(axes: pd.Series, x: float, y: float) -> float:
    """Where (x, y) projects along a piece's axis: 0 at either end, 0.5 in the middle."""
    span = axes["hi"] - axes["lo"]
    if span <= 0:
        return 0.5
    t = ((x - axes["cx"]) * axes["ux"] + (y - axes["cy"]) * axes["uy"] - axes["lo"]) / span
    t = float(np.clip(t, 0, 1))
    return min(t, 1 - t)


def pair_table(ctx: FrameContext, labels: np.ndarray, radius: int = 50) -> pd.DataFrame:
    """One row per pair of instances within `radius` px: PAIR_FEATURES plus frame, a, b."""
    near = pairs(labels, radius)
    if not len(near):
        return pd.DataFrame(columns=["frame", "a", "b", *PAIR_FEATURES])
    axes = component_axes(labels, int(labels.max()))
    exclude = cv2.dilate((labels > 0).astype(np.uint8), np.ones((7, 7), np.uint8)) > 0
    rows = []
    for p in near.itertuples(index=False):
        A, B = axes.iloc[p.a - 1], axes.iloc[p.b - 1]
        a_large = A["area"] >= B["area"]
        L, S = (A, B) if a_large else (B, A)
        pl, ps = ((p.ax, p.ay), (p.bx, p.by)) if a_large else ((p.bx, p.by), (p.ax, p.ay))
        gap = float(np.arctan2(p.by - p.ay, p.bx - p.ax) % np.pi)
        mid_x, mid_y = (p.ax + p.bx) // 2, (p.ay + p.by) // 2
        flat = path_values(ctx.flat, pl, ps).astype(float)
        prob = path_values(ctx.prob, pl, ps).astype(float) / 255
        fibril, coherence = fibril_direction(ctx.flat, mid_x, mid_y, exclude)
        rows.append(
            {
                "frame": ctx.name,
                "a": p.a,
                "b": p.b,
                "dist": p.dist,
                "angle_diff": float(line_angle(L["angle"], S["angle"])),
                "gap_large": float(line_angle(gap, L["angle"])),
                "gap_small": float(line_angle(gap, S["angle"])),
                "end_large": end_position(L, *pl),
                "end_small": end_position(S, *ps),
                "path_dark": float(1 - flat.mean() / REL_SCALE),
                "path_prob_mean": float(prob.mean()),
                "path_prob_min": float(prob.min()),
                "fibril_angle": float(line_angle(fibril, gap)),
                "fibril_coherence": coherence,
                "log_area_large": float(np.log(L["area"])),
                "log_area_small": float(np.log(S["area"])),
                "elong_large": L["elongation"],
                "elong_small": S["elongation"],
                "limb": float(np.hypot(mid_x - ctx.centre[0], mid_y - ctx.centre[1]) / ctx.radius)
                if ctx.radius
                else np.nan,
            }
        )
    return pd.DataFrame(rows, columns=["frame", "a", "b", *PAIR_FEATURES])


def same_filament(
    gt_crops_per_annotator: list[list[InstanceCrop]],
    labels: np.ndarray,
    areas: np.ndarray,
    table: pd.DataFrame,
    share: float = 0.3,
) -> tuple[np.ndarray, np.ndarray]:
    """(same, different) per pair: for some annotator, `share` or more of both pieces lies in one filament
    (same), or in two different filaments of theirs (different, and never same)."""
    same = np.zeros(len(table), bool)
    different = np.zeros(len(table), bool)
    if not len(table):
        return same, different
    a, b = table["a"].to_numpy(int) - 1, table["b"].to_numpy(int) - 1
    for crops in gt_crops_per_annotator:
        if not crops:
            continue
        intersection, _ = overlaps(crops, labels, areas)
        inside = intersection / np.maximum(areas, 1)[None, :] >= share  # (n_gt, n_pred)
        same |= (inside[:, a] & inside[:, b]).any(axis=0)
        different |= inside[:, a].any(axis=0) & inside[:, b].any(axis=0)
    return same, different & ~same


def join_pairs(labels: np.ndarray, table: pd.DataFrame, decide: np.ndarray):
    """Merge the pairs of `table` where `decide` is True (transitively); renumber largest first."""
    chosen = table.loc[np.asarray(decide, bool), ["a", "b"]].to_numpy(int)
    return merge(labels, [tuple(link) for link in chosen])


def aligned_end_to_end(table: pd.DataFrame, max_angle: float = np.radians(30), max_end: float = 0.15) -> np.ndarray:
    """Pairs that continue each other along the spine: the gap leaves the larger piece at its end along its
    axis, and also follows the smaller piece's axis when that piece is elongated enough to have one."""
    large = (table["elong_large"] >= ORIENTED) & (table["gap_large"] <= max_angle) & (table["end_large"] <= max_end)
    small = (table["elong_small"] < ORIENTED) | (table["gap_small"] <= max_angle)
    return (large & small).to_numpy()


def join_rule(table: pd.DataFrame, near: int = 10, far: int = 50) -> np.ndarray:
    """The rule of `join_lab.py`: pieces that continue each other end to end along the axis join up to `far`
    px apart, any two pieces up to `near` px (barbs, blobs)."""
    return ((table["dist"] <= near) | ((table["dist"] <= far) & aligned_end_to_end(table))).to_numpy()


def rescue(ctx: FrameContext, labels: np.ndarray, keep: np.ndarray, radius: int = 50):
    """Kept instances joined by `join_rule`, with each other and with dropped instances that continue them.

    `labels` holds every candidate 1..K before the filter and `keep` (bool[K]) the filter's decision. A dropped
    piece comes back only as part of a group that contains a kept one; groups of dropped pieces stay dropped.
    """
    keep = np.asarray(keep, bool)
    assert int(labels.max()) == len(keep), "one decision per candidate"
    table = pair_table(ctx, labels, radius)
    if len(table):
        a, b = table["a"].to_numpy(int) - 1, table["b"].to_numpy(int) - 1
        joined, areas, boxes = join_pairs(labels, table, join_rule(table) & (keep[a] | keep[b]))
    else:
        joined, areas, boxes = merge(labels, [])
    has_kept = np.zeros(len(areas) + 1, bool)
    has_kept[np.unique(joined[np.r_[False, keep][labels]])] = True
    has_kept[0] = False
    return select(joined, areas, boxes, has_kept[1:])

In [ ]:
%%writefile postprocess/pipeline.py
"""The post-processing of the best submission: candidates from the probability map, joined into whole
filaments, then filtered, with the filter and its cutoff learnt from out-of-fold frames.

1. **Candidates** (`A0`): hysteresis threshold 0.5 / 0.3 and components of at least 300 px.
2. **Join** (`joining.join_rule`): pieces that continue each other end to end along their axis join up to
   50 px apart, any two pieces up to 10 px (barbs). On 707 OOF frames this raised PQ by about +0.01.
3. **Filter** (`filtering.ComponentFilter`): gradient boosting on shape, darkness, limb distance and the
   network's confidence decides, per joined filament, whether it matched any annotator. Its cutoff is the
   one with the best PQ on out-of-fold filter scores.

Works on any model's maps: `fit` needs the maps of frames the model never saw (out of fold) with their
truth, and the fitted `Postprocess` is then applied to test maps of the same model, so both look alike.

    contexts = {f: frame_context(f, image, prob) for ...}
    post, oof = fit(contexts, truth, train_files)
    labels, areas, boxes = post(contexts[test_file])
"""

from dataclasses import dataclass, field

import numpy as np
import pandas as pd

from error_analysis.instances import FrameContext
from postprocess.filtering import ComponentFilter, component_table, matched_by_any_annotator, out_of_fold_scores
from postprocess.instances import cut, instances, select
from postprocess.joining import join_pairs, join_rule, pair_table
from postprocess.tuning import best_cutoff
from scoring.metric import InstanceCrop

A0 = {"threshold": 0.5, "min_area": 300, "low": 0.3}
PAIR_RADIUS = 50  # pairs further apart are never joined (join_rule's longest reach)


def candidates(ctx: FrameContext, params: dict = A0, join: bool = True):
    """(labels, areas, boxes) before the filter: `instances` of `ctx.prob`, joined when `join`.

    Labels are uint8 (at most `MAX_INSTANCES` per frame) so the candidates of 707 frames fit in memory.
    """
    labels, areas, boxes = instances(ctx.prob, **params)
    if join:
        table = pair_table(ctx, labels, PAIR_RADIUS)
        if len(table):
            labels, areas, boxes = join_pairs(labels, table, join_rule(table))
    return labels.astype(np.uint8), areas, boxes


@dataclass
class Postprocess:
    """A fitted post-processing: `post(ctx)` -> (labels, areas, boxes) of the kept filaments."""

    filter: ComponentFilter
    params: dict = field(default_factory=lambda: dict(A0))
    join: bool = True
    mask_cut: float | None = None  # kept filaments reduced to their pixels with probability >= mask_cut

    def __call__(self, ctx: FrameContext):
        inst = candidates(ctx, self.params, self.join)
        if not len(inst[1]):
            return inst
        kept = select(*inst, self.filter.keep(component_table(ctx, inst[0], inst[2])))
        return cut(*kept, ctx.prob, self.mask_cut) if self.mask_cut is not None else kept


@dataclass
class OutOfFold:
    """How the fitted post-processing did on the frames it was fitted on, with out-of-fold filter scores."""

    terms: pd.DataFrame  # per-frame PQ terms (`tuning.frame_terms`)
    cutoff: float
    kept_per_frame: float


def fit(
    contexts: dict[str, FrameContext],
    truth: dict[str, list[list[InstanceCrop]]],
    files: list[str],
    params: dict = A0,
    join: bool = True,
) -> tuple[Postprocess, OutOfFold]:
    """Fit the filter on the candidates of `files` and choose its cutoff on their out-of-fold PQ.

    `contexts[f].prob` must come from a model that never saw frame f; `truth[f]` lists the ground truth per
    annotator. The returned filter is refitted on all of `files`; the `OutOfFold` scores are honest.
    """
    raw = {f: candidates(contexts[f], params, join) for f in files}
    table = pd.concat(
        [component_table(contexts[f], raw[f][0], raw[f][2]) for f in files if len(raw[f][1])], ignore_index=True
    )
    target = np.concatenate([matched_by_any_annotator(truth[f], raw[f][0], raw[f][1]) for f in files])
    if not 0 < target.sum() < len(target):
        raise ValueError(
            f"the filter needs both matched and unmatched candidates; {target.sum()} of {len(target)} match"
        )
    scores = out_of_fold_scores(table, target)
    cutoff, terms = best_cutoff(scores, raw, table, truth, files)
    post = Postprocess(ComponentFilter(cutoff=cutoff).fit(table, target), dict(params), join)
    return post, OutOfFold(terms, cutoff, float((scores >= cutoff).sum()) / len(files))

In [ ]:
%%writefile postprocess/tuning.py
"""The filter's cutoff, chosen on Panoptic Quality.

PQ is a ratio of sums over the whole validation set, sum(IoU) / (TP + FP/2 + FN/2), so it is kept here as
per-frame terms (tp, fp, fn, iou_sum). Any subset or resample of frames is then scored by summing its terms.
"""

import numpy as np
import pandas as pd

from postprocess.instances import select
from scoring.metric import PanopticQuality

CUTOFFS = np.round(np.arange(0.2, 0.81, 0.05), 2)  # the grid of predict.py
TERMS = ["tp", "fp", "fn", "iou_sum"]


def frame_terms(raw: dict, table: pd.DataFrame, keep: np.ndarray, truth: dict, files: list[str]) -> pd.DataFrame:
    """One row per frame (in `files` order): PQ terms summed over its annotators, keeping components where `keep`.

    `raw[f]` is (labels, areas, boxes) from `instances`; `table` has one row per component in that order,
    frame after frame, as `component_table` builds it; `truth[f]` lists the ground truth per annotator.
    """
    keep = np.asarray(keep, dtype=bool)
    frames = table["frame"].to_numpy()
    rows = []
    for f in files:
        labels, areas, _ = select(*raw[f], keep[frames == f])
        pq = PanopticQuality()
        for gt_crops in truth[f]:
            pq.update(gt_crops, labels, areas)
        rows.append({"frame": f, "tp": pq.tp, "fp": pq.fp, "fn": pq.fn, "iou_sum": pq.iou_sum})
    return pd.DataFrame(rows, columns=["frame", *TERMS])


def pq_of(terms: pd.DataFrame) -> float:
    """PQ of the frames in `terms`: sums first, ratio last (the mean of per-frame PQs is a different number)."""
    tp, fp, fn, iou_sum = (float(terms[c].sum()) for c in TERMS)
    denominator = tp + 0.5 * fp + 0.5 * fn
    return iou_sum / denominator if denominator > 0 else 0.0


def best_cutoff(
    scores: np.ndarray, raw: dict, table: pd.DataFrame, truth: dict, files: list[str], cutoffs=CUTOFFS
) -> tuple[float, pd.DataFrame]:
    """(cutoff, frame terms) of the cutoff on the filter scores with the highest PQ; ties go to the lowest cutoff."""
    best = None
    for cutoff in cutoffs:
        terms = frame_terms(raw, table, scores >= cutoff, truth, files)
        if best is None or pq_of(terms) > pq_of(best[1]):
            best = (float(cutoff), terms)
    return best

In [ ]:
%%writefile preprocess/__init__.py
"""Image preprocessing shared by the approaches: reading frames, solar-disk geometry, limb darkening."""

In [ ]:
%%writefile preprocess/disk.py
"""Solar-disk geometry and limb-darkening removal for full-disk H-alpha frames."""

import cv2
import numpy as np

REL_SCALE = 200  # `flatten` stores image / background as uint8, with the background level at REL_SCALE
OFF_DISK = 255  # value `flatten` assigns outside the (shrunk) disk


def disk_geometry(image: np.ndarray) -> tuple[float, float, float]:
    """(centre x, centre y, radius) of the solar disk in pixels; radius 0 when no disk is found.

    The sky around the disk is grey rather than black in many frames, hence Otsu instead of a fixed
    threshold. Centre and radius come from the blob's centroid and area, which ignores prominences.
    """
    blurred = cv2.GaussianBlur(image, (0, 0), 5)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, _, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n < 2:
        return image.shape[1] / 2, image.shape[0] / 2, 0.0
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return float(centroids[k][0]), float(centroids[k][1]), float(np.sqrt(stats[k, cv2.CC_STAT_AREA] / np.pi))


def disk_mask(image: np.ndarray, shrink: float = 1.0) -> np.ndarray:
    """uint8 {0, 1} mask of the solar disk, its radius scaled by `shrink`."""
    cx, cy, radius = disk_geometry(image)
    mask = np.zeros_like(image, dtype=np.uint8)
    if radius > 0:
        cv2.circle(mask, (round(cx), round(cy)), int(radius * shrink), 1, -1)
    return mask


def flatten(image: np.ndarray, shrink: float = 0.96, background_sigma: float = 40, smooth_sigma: float = 2):
    """Remove limb darkening: uint8 map of REL_SCALE * image / smooth background, OFF_DISK off-disk.

    The background is smooth, so it is estimated at 1/8 resolution. It is a normalised convolution
    (blurred image / blurred disk mask), which keeps the dark sky from bleeding into the limb.
    `shrink` drops the outermost ring, where the limb itself would read as a dark structure.
    """
    size = image.shape[0]
    disk = disk_mask(image, shrink).astype(np.float32)
    image_f = image.astype(np.float32)  # uint8 arithmetic would wrap around
    small = (size // 8, size // 8)
    weight = cv2.GaussianBlur(cv2.resize(disk, small, interpolation=cv2.INTER_AREA), (0, 0), background_sigma / 8)
    blurred = cv2.GaussianBlur(
        cv2.resize(image_f * disk, small, interpolation=cv2.INTER_AREA), (0, 0), background_sigma / 8
    )
    background = cv2.resize(blurred / np.maximum(weight, 1e-3), (size, size), interpolation=cv2.INTER_LINEAR)
    # filaments are ~10-20 px wide, so a light blur only tames the chromospheric texture
    smooth = cv2.GaussianBlur(image_f, (0, 0), smooth_sigma) if smooth_sigma > 0 else image_f
    rel = np.clip(REL_SCALE * smooth / np.maximum(background, 1.0), 0, 255).astype(np.uint8)
    rel[disk == 0] = OFF_DISK
    return rel

In [ ]:
%%writefile preprocess/images.py
"""Reading the competition's JPEG frames."""

from pathlib import Path

import cv2
import numpy as np


def read_gray(path: str | Path) -> np.ndarray:
    """uint8 (H, W). The frames are grayscale; reading them as colour would only triplicate the data."""
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f"cannot read {path}")
    return image

In [ ]:
%%writefile scoring/__init__.py
"""How a submission is scored: the mask encoding and CSV the hosts expect, their Panoptic Quality
metric, and a format check to run before spending one of the five daily submissions."""

In [ ]:
%%writefile scoring/check.py
"""Sanity-check a submission CSV against the competition's format before spending a submission."""

import re
from pathlib import Path

import numpy as np
import pandas as pd
from pycocotools import mask as mask_util

from dataset import IMAGE_SIZE, N_TEST_IMAGES
from scoring import rle
from scoring.submission import COLUMNS

ID_PATTERN = re.compile(r"^\d{14}[A-Z][a-z]_\w+$")  # e.g. 20150125172714Mh_1


def check(path: Path) -> list[str]:
    """Print a summary of the file and return the list of errors (empty = safe to submit)."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if list(df.columns) != COLUMNS:
        return [f"columns must be {COLUMNS}, got {list(df.columns)}"]
    errors = []
    if df["filament_id"].duplicated().any():
        errors.append(f"{int(df['filament_id'].duplicated().sum())} duplicated filament_id")
    bad_ids = df.loc[~df["filament_id"].str.match(ID_PATTERN), "filament_id"]
    if len(bad_ids):
        errors.append(f"{len(bad_ids)} malformed filament_id, e.g. {bad_ids.iloc[0]!r}")
    valid = df["segmentation_rle"].map(rle.is_valid)
    if not valid.all():
        errors.append(f"{int((~valid).sum())} invalid RLE strings, e.g. row {int(valid.idxmin())}")
        df = df[valid]

    areas = []
    overlap_images = 0
    image_ids = df["filament_id"].str.split("_", n=1).str[0]
    for _, group in df.groupby(image_ids):
        rles = [{"size": [IMAGE_SIZE, IMAGE_SIZE], "counts": c.encode("ascii")} for c in group["segmentation_rle"]]
        group_areas = mask_util.area(rles)
        areas.extend(group_areas.tolist())
        if len(rles) > 1 and int(mask_util.area(mask_util.merge(rles))) < int(group_areas.sum()):
            overlap_images += 1
    if 0 in areas:
        errors.append(f"{areas.count(0)} empty masks")

    n_images = image_ids.nunique()
    print(f"{path}: {len(df)} filaments over {n_images}/{N_TEST_IMAGES} test images")
    if areas:
        q = np.percentile(areas, [0, 50, 100]).astype(int)
        print(f"  filaments/image: {len(df) / max(n_images, 1):.1f} | area px min/median/max: {q[0]}/{q[1]}/{q[2]}")
    if overlap_images:
        print(f"  warning: predicted filaments overlap each other in {overlap_images} images")
    if n_images < N_TEST_IMAGES:
        print(f"  warning: {N_TEST_IMAGES - n_images} test images have no prediction")
    for error in errors:
        print("  ERROR:", error)
    print("  OK" if not errors else f"  {len(errors)} error(s)")
    return errors

In [ ]:
%%writefile scoring/metric.py
"""Panoptic Quality as defined by the hosts' self-evaluation notebook.

    PQ = sum of IoU over true-positive pairs / (|TP| + 0.5 |FP| + 0.5 |FN|)

A ground-truth/prediction pair is a true positive when IoU > 0.5; unmatched predictions are false
positives and unmatched ground-truth filaments are false negatives. Scores are accumulated over
annotator-images, i.e. the same prediction is compared with every annotator's set for that image.

Predictions are passed as a label map, so they cannot overlap and a filament matches at most one
prediction. Overlaps are counted inside each ground-truth bounding box only, which makes evaluating
a 2048 x 2048 frame a matter of milliseconds.
"""

import numpy as np
from pycocotools import mask as mask_util

IOU_THRESHOLD = 0.5

InstanceCrop = tuple[int, int, np.ndarray]  # x, y, boolean mask of the bounding box


def instance_crops(rles: list[dict]) -> list[InstanceCrop]:
    """Ground-truth RLEs -> bounding-box crops, the form `PanopticQuality.update` consumes."""
    crops = []
    for rle in rles:
        x, y, w, h = (int(v) for v in mask_util.toBbox(rle))
        if w > 0 and h > 0:
            crops.append((x, y, mask_util.decode(rle)[y : y + h, x : x + w].astype(bool)))
    return crops


class PanopticQuality:
    def __init__(self):
        self.iou_sum, self.tp, self.fp, self.fn = 0.0, 0, 0, 0

    def update(self, gt_crops: list[InstanceCrop], labels: np.ndarray, pred_areas: np.ndarray) -> None:
        """Add one annotator-image. `labels` holds 0 for background and 1..K for the K predictions."""
        n_pred = len(pred_areas)
        if not gt_crops or n_pred == 0:
            self.fp += n_pred
            self.fn += len(gt_crops)
            return
        pred_hit = np.zeros(n_pred, dtype=bool)
        for x, y, crop in gt_crops:
            h, w = crop.shape
            intersection = np.bincount(labels[y : y + h, x : x + w][crop], minlength=n_pred + 1)[1:]
            iou = intersection / (crop.sum() + pred_areas - intersection)
            best = int(iou.argmax())
            if iou[best] > IOU_THRESHOLD:
                self.tp += 1
                self.iou_sum += float(iou[best])
                pred_hit[best] = True
            else:
                self.fn += 1
        self.fp += int((~pred_hit).sum())

    @property
    def value(self) -> float:
        denominator = self.tp + 0.5 * self.fp + 0.5 * self.fn
        return self.iou_sum / denominator if denominator > 0 else 0.0

    def summary(self) -> dict:
        """PQ with its factors: SQ (mean IoU of matches) and RQ (an F1 over instances)."""
        sq = self.iou_sum / self.tp if self.tp else 0.0
        return {
            "pq": round(self.value, 4),
            "sq": round(sq, 4),
            "rq": round(self.value / sq, 4) if sq else 0.0,
            "tp": self.tp,
            "fp": self.fp,
            "fn": self.fn,
        }

In [ ]:
%%writefile scoring/rle.py
"""COCO run-length encoding helpers. The submission stores only the compressed `counts` string."""

import numpy as np
from pycocotools import mask as mask_util

from dataset import IMAGE_SIZE


def encode(mask: np.ndarray) -> str:
    """Binary (H, W) mask -> compressed RLE counts."""
    return mask_util.encode(np.asfortranarray(mask.astype(np.uint8)))["counts"].decode("ascii")


def decode(counts: str, size: int = IMAGE_SIZE) -> np.ndarray:
    """Compressed RLE counts -> uint8 (size, size) mask."""
    return mask_util.decode({"size": [size, size], "counts": counts.encode("ascii")})


def encode_component(labels: np.ndarray, label: int, box: tuple[int, int, int, int]) -> str:
    """Encode one instance of a label map, touching only its bounding box (x, y, w, h)."""
    x, y, w, h = box
    mask = np.zeros(labels.shape, dtype=np.uint8, order="F")
    mask[y : y + h, x : x + w] = labels[y : y + h, x : x + w] == label
    return mask_util.encode(mask)["counts"].decode("ascii")


def is_valid(counts: str, size: int = IMAGE_SIZE) -> bool:
    """True if `counts` is well-formed and its runs cover the image exactly.

    pycocotools decodes garbage without complaint, so the string is parsed here: each number is a
    little-endian sequence of 5-bit groups (bit 5 = continuation, bit 4 of the last group = sign),
    and from the fourth number on values are deltas against the number two positions earlier.
    """
    runs, pos = [], 0
    try:
        while pos < len(counts):
            value, group, more = 0, 0, True
            while more:
                char = ord(counts[pos]) - 48
                if not 0 <= char < 64:
                    return False
                value |= (char & 0x1F) << (5 * group)
                more = bool(char & 0x20)
                pos += 1
                group += 1
                if not more and (char & 0x10):
                    value |= -1 << (5 * group)
            if len(runs) > 2:
                value += runs[-2]
            if value < 0:
                return False
            runs.append(value)
    except IndexError:  # the string ended in the middle of a number
        return False
    return sum(runs) == size * size

In [ ]:
%%writefile scoring/submission.py
"""Submission file: one row per predicted filament, `filament_id` = `<image id>_<n>`."""

from pathlib import Path

import numpy as np
import pandas as pd

from scoring import rle

COLUMNS = ["filament_id", "segmentation_rle"]


def rows_from_label_map(image_id: str, labels: np.ndarray, boxes: np.ndarray) -> list[dict]:
    """Rows for the instances 1..K of a label map; `boxes[k - 1]` is instance k as (x, y, w, h)."""
    return [
        {"filament_id": f"{image_id}_{k}", "segmentation_rle": rle.encode_component(labels, k, tuple(box))}
        for k, box in enumerate(boxes, start=1)
    ]


def write(rows: list[dict], path: str | Path) -> pd.DataFrame:
    frame = pd.DataFrame(rows, columns=COLUMNS)
    frame.to_csv(path, index=False)
    return frame

In [ ]:
%%writefile cnn_baseline/__init__.py
"""CNN baseline: a plain U-Net trained from scratch on tiles, predicting a filament probability map;
instances are its connected components. The simplest learned approach, to see how far it gets."""

In [ ]:
%%writefile cnn_baseline/data.py
"""Frames as network input, soft filament targets, and tile sampling for training."""

import random
from pathlib import Path

import cv2
import numpy as np
import torch
from pycocotools import mask as mask_util
from torch.utils.data import Dataset

from dataset.annotations import Annotations
from preprocess.disk import OFF_DISK, flatten
from preprocess.images import read_gray

OFF_DISK_VALUE = OFF_DISK / 255.0  # what the flattened channel holds outside the disk


def split_files(files: list[str], validation_fraction: float, seed: int) -> tuple[list[str], list[str]]:
    """(training, validation) file names: a deterministic shuffle of the physical frames.

    Splitting by frame keeps the up-to-three annotation sets of one frame on the same side.
    """
    files = sorted(files)
    random.Random(seed).shuffle(files)
    n_val = max(1, round(len(files) * validation_fraction))
    return files[n_val:], files[:n_val]


def fold_files(files: list[str], n_folds: int, seed: int) -> list[list[str]]:
    """`n_folds` disjoint groups of physical frames covering all of `files` (the shuffle of `split_files`).

    Every frame is held out exactly once, so out-of-fold predictions cover the whole training set.
    """
    files = sorted(files)
    random.Random(seed).shuffle(files)
    return [files[k::n_folds] for k in range(n_folds)]


def channels(image: np.ndarray) -> np.ndarray:
    """float32 (2, H, W): the raw frame and its limb-darkening-corrected version, both scaled to [0, 1]."""
    return np.stack([image, flatten(image)]).astype(np.float32) / 255.0


def annotator_masks(annotations: Annotations, file_name: str) -> np.ndarray:
    """bool (A, H, W): the filaments of each of the frame's A annotation sets (an empty set gives an empty mask)."""
    per_annotator = annotations.rles_for_file(file_name)
    if not per_annotator:
        raise KeyError(f"no annotations for {file_name}")
    masks = [mask_util.decode(rles).any(axis=2) if rles else None for rles in per_annotator]
    shape = next(m.shape for m in masks if m is not None)
    return np.stack([m if m is not None else np.zeros(shape, bool) for m in masks])


def soft_target(annotations: Annotations, file_name: str) -> np.ndarray:
    """float32 (H, W) in [0, 1]: the fraction of annotators who marked each pixel as filament.

    Up to three annotators labelled the same frame independently; averaging them keeps the
    disagreement as a soft label instead of arbitrarily trusting one of them.
    """
    return annotator_masks(annotations, file_name).mean(axis=0, dtype=np.float32)


LABELS = ("mean", "sample")


class FilamentFrames(Dataset):
    """One item = one physical frame.

    Training mode returns `tiles_per_frame` random tiles from it: (K, 2, tile, tile) inputs and
    (K, tile, tile) targets. Half of the tiles are centred on a filament pixel, because filaments
    cover 0.4 % of the disk and purely random tiles would mostly be empty. Evaluation mode returns
    the whole frame, (2, H, W) and (H, W).

    `label` "mean" trains on the fraction of annotators marking each pixel; "sample" on one annotation set,
    drawn anew each time the frame is visited. Averaging thins a filament only one of three annotators drew
    to a 1/3 target; sampling keeps every annotator's filaments whole, with the disagreement in the draws.
    """

    def __init__(
        self,
        files: list[str],
        image_dir: Path,
        annotations: Annotations,
        tile: int = 512,
        tiles_per_frame: int = 4,
        train: bool = True,
        label: str = "mean",
    ):
        if label not in LABELS:
            raise ValueError(f"label must be one of {LABELS}, not {label!r}")
        self.files, self.image_dir, self.annotations = files, Path(image_dir), annotations
        self.tile, self.tiles_per_frame, self.train, self.label = tile, tiles_per_frame, train, label

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, index: int):
        x = channels(read_gray(self.image_dir / self.files[index]))
        masks = annotator_masks(self.annotations, self.files[index])
        if not self.train:
            return torch.from_numpy(x), torch.from_numpy(masks.mean(axis=0, dtype=np.float32))
        # torch seeds each DataLoader worker differently every epoch, so tiles differ across epochs
        rng = np.random.default_rng(int(torch.randint(0, 2**31, (1,))))
        if self.label == "sample":
            y = masks[rng.integers(len(masks))].astype(np.float32)
        else:
            y = masks.mean(axis=0, dtype=np.float32)
        tiles = [self.sample_tile(x, y, rng) for _ in range(self.tiles_per_frame)]
        return torch.from_numpy(np.stack([t[0] for t in tiles])), torch.from_numpy(np.stack([t[1] for t in tiles]))

    def sample_tile(self, x: np.ndarray, y: np.ndarray, rng: np.random.Generator):
        positives = np.flatnonzero(y.ravel() > 0.5)
        if len(positives) and rng.random() < 0.5:
            centre = positives[rng.integers(len(positives))]
        else:
            on_disk = np.flatnonzero(x[1].ravel() < OFF_DISK_VALUE)
            centre = on_disk[rng.integers(len(on_disk))] if len(on_disk) else rng.integers(y.size)
        height, width = y.shape
        row, col = divmod(int(centre), width)
        top = int(np.clip(row - self.tile // 2, 0, height - self.tile))
        left = int(np.clip(col - self.tile // 2, 0, width - self.tile))
        tile_x = x[:, top : top + self.tile, left : left + self.tile]
        tile_y = y[top : top + self.tile, left : left + self.tile]
        # only masks are scored, so every rotation and flip is a valid augmentation
        k = int(rng.integers(4))
        tile_x, tile_y = np.rot90(tile_x, k, axes=(1, 2)), np.rot90(tile_y, k)
        if rng.random() < 0.5:
            tile_x, tile_y = tile_x[:, :, ::-1], tile_y[:, ::-1]
        return np.ascontiguousarray(tile_x), np.ascontiguousarray(tile_y)


def worker_init(_: int) -> None:
    """OpenCV's own thread pool and forked DataLoader workers do not mix; one thread per worker is plenty."""
    cv2.setNumThreads(0)


def collate_tiles(batch):
    """Merge the per-frame tile stacks into one batch: (B*K, 2, tile, tile), (B*K, tile, tile)."""
    xs, ys = zip(*batch)
    return torch.cat(xs), torch.cat(ys)

In [ ]:
%%writefile cnn_baseline/training.py
"""Training loop and full-frame inference for a binary segmentation network."""

import os
import time
from collections.abc import Iterable, Iterator

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader


def pick_device() -> torch.device:
    """CUDA if there is one; locally fall back to Apple's MPS or the CPU.

    On Kaggle a missing GPU is an error, not a fallback: an account without phone verification (or
    without GPU quota) gets a CPU machine for a GPU notebook without any warning, and this U-Net then
    needs ~3.5 h per epoch instead of minutes.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        raise RuntimeError(
            "Kaggle gave this notebook no GPU. Check that the account is phone-verified and has GPU quota "
            "left (Settings on kaggle.com); stopping instead of training on the CPU."
        )
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def bce_dice_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """BCE for per-pixel accuracy plus soft Dice for the 0.4 % of pixels that are filament.

    Both accept soft targets, so annotator disagreement flows into the loss as fractional labels.
    """
    logits, target = logits.squeeze(1).float(), target.float()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    intersection = (prob * target).sum(dim=(1, 2))
    dice = 1 - (2 * intersection + 1) / (prob.sum(dim=(1, 2)) + target.sum(dim=(1, 2)) + 1)
    return bce + dice.mean()


def boundary_band(target: torch.Tensor, width: int) -> torch.Tensor:
    """bool (B, H, W): pixels within `width` px of a place where the (soft) target changes value.

    Morphological gradient by max pooling; on the mean of several annotators it covers each one's border.
    """
    t = target.unsqueeze(1).float()
    k = 2 * width + 1
    dilated = F.max_pool2d(t, k, stride=1, padding=width)
    eroded = -F.max_pool2d(-t, k, stride=1, padding=width)
    return (dilated - eroded).squeeze(1) > 0


def boundary_bce_dice_loss(
    logits: torch.Tensor, target: torch.Tensor, weight: float = 4.0, width: int = 3
) -> torch.Tensor:
    """`bce_dice_loss` with the BCE of pixels near a filament's border counted `1 + weight` times.

    Near misses (IoU just under 0.5) lose their filament's border and faint ends, not its core; weighting
    the border band moves the gradient there. The BCE is a weighted mean, so its scale stays comparable.
    """
    logits, target = logits.squeeze(1).float(), target.float()
    w = 1 + weight * boundary_band(target, width).float()
    bce = (F.binary_cross_entropy_with_logits(logits, target, reduction="none") * w).sum() / w.sum()
    prob = torch.sigmoid(logits)
    intersection = (prob * target).sum(dim=(1, 2))
    dice = 1 - (2 * intersection + 1) / (prob.sum(dim=(1, 2)) + target.sum(dim=(1, 2)) + 1)
    return bce + dice.mean()


def soft_skeleton(image: torch.Tensor, iterations: int = 30) -> torch.Tensor:
    """(B, H, W) skeleton of a (soft) mask by iterated min / max pooling, as clDice's soft-skeletonisation.

    On a binary mask it is close to the morphological skeleton; on the mean of several annotators each
    skeleton pixel keeps the fraction that drew it. `iterations` must reach half the widest filament (a 60 px wide blob needs 30).
    """

    def erode(t):
        return torch.minimum(-F.max_pool2d(-t, (3, 1), 1, (1, 0)), -F.max_pool2d(-t, (1, 3), 1, (0, 1)))

    def opened(t):
        return F.max_pool2d(erode(t), 3, 1, 1)

    t = image.unsqueeze(1).float()
    skeleton = F.relu(t - opened(t))
    for _ in range(iterations):
        t = erode(t)
        delta = F.relu(t - opened(t))
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton.squeeze(1)


def skeleton_recall_bce_dice_loss(
    logits: torch.Tensor, target: torch.Tensor, weight: float = 1.0, level: float | None = None
) -> torch.Tensor:
    """`bce_dice_loss` plus `weight` x (1 - recall of the target's skeleton), as Skeleton Recall (Kirchhoff
    et al., 2024).

    A filament whose faint middle falls below 0.5 comes out in pieces or as a fragment; BCE and Dice barely
    notice the few pixels of the gap, but they are a large share of the filament's centre line. The skeleton
    is thickened to 3 px ("tubed") so a prediction a pixel off still covers it; tiles without filament add 0.

    With `level`, the skeleton is that of the pixels at least that share of the annotators drew (0.5: a
    majority) instead of the soft target's: recall rewards p = 1 on every skeleton pixel, so a skeleton of
    filaments one annotator in three drew pulls the map up where they disagree (`zoo/swin_unet_skeleton`).
    """
    with torch.no_grad():
        drawn = target >= level if level is not None else target > 0
        source = drawn.float() if level is not None else target
        skeleton = F.max_pool2d(soft_skeleton(source).unsqueeze(1), 3, 1, 1).squeeze(1).clamp(max=1)
        skeleton = skeleton * drawn  # the tube stays inside the drawn filament
    prob = torch.sigmoid(logits.squeeze(1).float())
    length = skeleton.sum(dim=(1, 2))
    recall = (prob * skeleton).sum(dim=(1, 2)) / length.clamp(min=1)
    has_filament = (length > 0).float()
    missed = ((1 - recall) * has_filament).sum() / has_filament.sum().clamp(min=1)
    return bce_dice_loss(logits, target) + weight * missed


def parameter_groups(model: nn.Module, lr: float, pretrained_lr: float | None) -> list[dict]:
    """One group at `lr`, or -- for a model with `pretrained_parameters()` and a `pretrained_lr` -- the
    pretrained part at `pretrained_lr` and the rest (decoder, head) at `lr`."""
    if pretrained_lr is None or not hasattr(model, "pretrained_parameters"):
        return [{"params": list(model.parameters()), "lr": lr}]
    pretrained = list(model.pretrained_parameters())
    ids = {id(p) for p in pretrained}
    rest = [p for p in model.parameters() if id(p) not in ids]
    return [{"params": pretrained, "lr": pretrained_lr}, {"params": rest, "lr": lr}]


def fit(
    model: nn.Module,
    loader: DataLoader,
    epochs: int,
    lr: float = 1e-3,
    device: torch.device | None = None,
    log=print,
    loss=bce_dice_loss,
    pretrained_lr: float | None = None,
    ema: float | None = None,
) -> list[float]:
    """AdamW + one-cycle schedule, mixed precision on CUDA. Returns the mean loss of each epoch.

    `loss(logits, target)` defaults to BCE + Dice; `pretrained_lr` sets a smaller peak learning rate for the
    pretrained encoder of a model that has one (see `parameter_groups`). With `ema` (a decay such as 0.998),
    an exponential moving average of the weights and buffers is kept after every step and copied into `model`
    at the end, so the model returned is the average of its last ~1 / (1 - ema) steps.
    """
    device = device or pick_device()
    model.to(device)
    groups = parameter_groups(model, lr, pretrained_lr)
    optimizer = torch.optim.AdamW(groups, lr=lr, weight_decay=1e-4)
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[g["lr"] for g in groups], total_steps=epochs * len(loader)
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    average = (
        torch.optim.swa_utils.AveragedModel(
            model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(ema), use_buffers=True
        )
        if ema
        else None
    )
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        start, total, steps = time.time(), 0.0, 0
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                value = loss(model(x), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(value).backward()
            scaler.step(optimizer)
            scaler.update()
            schedule.step()
            if average is not None:
                average.update_parameters(model)
            total, steps = total + value.item(), steps + 1
        history.append(total / max(steps, 1))
        log(
            f"epoch {epoch}/{epochs}  loss {history[-1]:.4f}  lr {schedule.get_last_lr()[0]:.2e}  {time.time() - start:.0f}s"
        )
    if average is not None:
        model.load_state_dict(average.module.state_dict())
    return history


@torch.no_grad()
def predict(
    model: nn.Module, frames: Iterable[torch.Tensor], device: torch.device | None = None
) -> Iterator[np.ndarray]:
    """Yield a uint8 (H, W) probability map (probability * 255) for each (2, H, W) frame."""
    device = device or pick_device()
    model.to(device).eval()
    use_amp = device.type == "cuda"
    for frame in frames:
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(frame.unsqueeze(0).to(device))
        prob = torch.sigmoid(logits.float()).squeeze().mul(255).round().to(torch.uint8)
        yield prob.cpu().numpy()


D4 = [(k, flip) for flip in (False, True) for k in range(4)]  # (quarter turns, mirrored); identity first


@torch.no_grad()
def predict_tta(
    model: nn.Module, frames: Iterable[torch.Tensor], device: torch.device | None = None, views: int = 8
) -> Iterator[tuple[np.ndarray, np.ndarray]]:
    """Yield (plain, averaged) uint8 maps per (2, H, W) frame: the identity view alone, as `predict` gives it,
    and the mean probability over the first `views` rotations / mirror images, each turned back first.

    A filament has no preferred orientation in the image, so the network should answer the same for every
    view; averaging removes the part of its output that depends on the view.
    """
    device = device or pick_device()
    model.to(device).eval()
    use_amp = device.type == "cuda"
    for frame in frames:
        plain, total = None, None
        for k, flip in D4[:views]:
            x = torch.rot90(frame.flip(-1) if flip else frame, k, dims=(-2, -1))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(x.unsqueeze(0).to(device))
            prob = torch.rot90(torch.sigmoid(logits.float()).squeeze(), -k, dims=(-2, -1))
            prob = prob.flip(-1) if flip else prob
            plain = prob if plain is None else plain
            total = prob.clone() if total is None else total + prob
        yield (
            plain.mul(255).round().to(torch.uint8).cpu().numpy(),
            total.div(views).mul(255).round().to(torch.uint8).cpu().numpy(),
        )

In [ ]:
%%writefile zoo/experiment.py
"""Five-fold training of one network / loss / label setting: out-of-fold maps for all 707 training frames.

Every experiment of the zoo goes through this one function, so they differ only in what `EXPERIMENTS`
says: the folds (`fold_files`, seed 0, as `cnn_baseline/cv.py`), tiles, augmentation, epochs, schedule
and inference are shared. Inference runs on 512 px tiles with Hann-window blending for every network,
the size the transformers were built for, so all maps are made the same way.

Outputs, with the experiment's name as prefix so several can be attached to one comparison notebook:
`<name>_oof_fold{k}.npz` (uint8 probability * 255 of fold k's frames), `<name>_test_fold{k}.npz` (fold k's
model on the test frames), `<name>_fold{k}.pt`, `<name>_folds.json`, `<name>_report.json` (losses, timings,
peak GPU memory and a quick PQ: A0's instances without the component filter).
"""

import json
import os
import random
import time
from functools import partial

import numpy as np
import torch
from torch.utils.data import DataLoader

from cnn_baseline.data import FilamentFrames, channels, collate_tiles, fold_files, worker_init
from cnn_baseline.training import (
    bce_dice_loss,
    boundary_bce_dice_loss,
    fit,
    pick_device,
    skeleton_recall_bce_dice_loss,
)
from dataset.annotations import load_annotations
from dataset.paths import find_data_root, output_dir
from postprocess.instances import instances
from postprocess.pipeline import A0
from preprocess.images import read_gray
from scoring.metric import PanopticQuality, instance_crops

QUICK = bool(os.environ.get("FILAMENT_QUICK"))  # a few frames, one epoch, small tiles and no downloads
SEED = 0
N_FOLDS = 5
TILE = 256 if QUICK else 512
STRIDE = TILE * 3 // 4
TILES_PER_FRAME = 4
FRAMES_PER_BATCH = 2 if QUICK else 4  # -> 16 tiles per step, as the baseline
EPOCHS = 1 if QUICK else 16
LR = 1e-3
PRETRAINED_LR = 1e-4  # ImageNet encoders: a tenth of the decoder's peak learning rate
WORKERS = 0 if QUICK else 3
BUDGET_SEC = 11 * 3600  # Kaggle stops a session at 12 h; no fold starts that would not finish before this

# name -> (network, loss, label); the first three differ from the baseline in one thing each
EXPERIMENTS = {
    "baseline": ("unet", "bce_dice", "mean"),
    "boundary_bce": ("unet", "boundary", "mean"),
    "annotator_sampling": ("unet", "bce_dice", "sample"),
    "edgeattnet": ("edgeattnet", "bce_dice", "mean"),
    "hrnet": ("hrnet_w18", "bce_dice", "mean"),
    "segformer": ("segformer", "bce_dice", "mean"),
    "swin_unet": ("swin_unet", "bce_dice", "mean"),
    "swin_unet_boundary": ("swin_unet", "boundary", "mean"),  # the two that helped, together
    "swin_unet_skeleton": ("swin_unet", "skeleton", "mean"),  # against filaments cut by faint stretches
    "swin_unet_skeleton_majority": ("swin_unet", "skeleton_majority", "mean"),  # the same, on agreed lines
    "swin_unet_boundary_seed1": ("swin_unet", "boundary", "mean"),  # a second seed, for a seed ensemble
    "swin_unet_boundary_ema": ("swin_unet", "boundary", "mean"),  # one model like the mean of several
}
# training seeds start here instead of SEED (the folds stay those of SEED, so out-of-fold maps line up)
TRAIN_SEEDS = {"swin_unet_boundary_seed1": 100}
# weights averaged over training (`fit`'s `ema` decay; 0.998 ~ the last 500 steps, 3.5 of 16 epochs)
EMA = {"swin_unet_boundary_ema": 0.998}
LOSSES = {
    "bce_dice": bce_dice_loss,
    "boundary": boundary_bce_dice_loss,
    "skeleton": skeleton_recall_bce_dice_loss,
    "skeleton_majority": partial(skeleton_recall_bce_dice_loss, weight=0.2, level=0.5),
}


def build(network: str) -> torch.nn.Module:
    """The network, with ImageNet weights where it has a pretrained encoder (none in quick runs)."""
    pretrained = not QUICK
    if network == "unet":
        from models.unet import UNet

        return UNet(in_channels=2, base=8 if QUICK else 32, depth=4)
    if network == "edgeattnet":
        from models.edgeattnet import EdgeAttNet

        return EdgeAttNet(in_channels=2, base=8 if QUICK else 32, depth=4)
    if network == "hrnet_w18":
        from models.pretrained_unet import PretrainedUNet

        return PretrainedUNet("hrnet_w18", pretrained=pretrained)
    if network == "segformer":
        from models.segformer import SegFormer

        return SegFormer(pretrained=pretrained)
    if network == "swin_unet":
        from models.swin_unet import SwinUNet

        return SwinUNet(pretrained=pretrained, tile=TILE)
    raise ValueError(f"unknown network {network!r}")


def starts(length: int, tile: int, stride: int) -> list[int]:
    return [0] if length <= tile else sorted({*range(0, length - tile + 1, stride), length - tile})


@torch.no_grad()
def predict_tiled(model, frame: torch.Tensor, device, tile: int = TILE, stride: int = STRIDE, batch: int = 8):
    """uint8 (H, W) probability * 255 of a (2, H, W) frame from overlapping tiles, blended with a Hann window."""
    model.to(device).eval()
    _, height, width = frame.shape
    window = torch.from_numpy(np.maximum(np.outer(np.hanning(tile), np.hanning(tile)), 1e-3).astype(np.float32))
    total = torch.zeros(height, width)
    weight = torch.zeros(height, width)
    corners = [(top, left) for top in starts(height, tile, stride) for left in starts(width, tile, stride)]
    for i in range(0, len(corners), batch):
        chunk = corners[i : i + batch]
        x = torch.stack([frame[:, t : t + tile, l : l + tile] for t, l in chunk]).to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            prob = torch.sigmoid(model(x).float())[:, 0].cpu()
        for (t, l), p in zip(chunk, prob):
            total[t : t + tile, l : l + tile] += p * window
            weight[t : t + tile, l : l + tile] += window
    return (total / weight).mul(255).round().to(torch.uint8).numpy()


# (quarter turns, mirrored) of the 8 views of the square; the first four are identity, flips and half turn
VIEWS = [(0, False), (0, True), (2, True), (2, False), (1, False), (1, True), (3, False), (3, True)]


def predict_views(model, frame: torch.Tensor, device, views: int = 8) -> np.ndarray:
    """uint8 (H, W) mean of `predict_tiled` over the first `views` flips and quarter turns, each turned back."""
    total = torch.zeros(frame.shape[1:])
    for turns, flip in VIEWS[:views]:
        view = torch.rot90(frame, turns, dims=(-2, -1))
        view = view.flip(-1) if flip else view
        prob = torch.from_numpy(predict_tiled(model, view.contiguous(), device)).float()
        prob = prob.flip(-1) if flip else prob
        total += torch.rot90(prob, -turns, dims=(-2, -1))
    return total.div(views).round().to(torch.uint8).numpy()


def train(name: str, train_files: list[str], train_dir, annotations, device, seed: int, log=print):
    """The experiment's network trained on `train_files` with its loss and label: (model, epoch losses)."""
    network, loss, label = EXPERIMENTS[name]
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    loader = DataLoader(
        FilamentFrames(train_files, train_dir, annotations, tile=TILE, tiles_per_frame=TILES_PER_FRAME, label=label),
        batch_size=FRAMES_PER_BATCH,
        shuffle=True,
        num_workers=WORKERS,
        collate_fn=collate_tiles,
        worker_init_fn=worker_init,
        pin_memory=device.type == "cuda",
        persistent_workers=WORKERS > 0,
        drop_last=True,
    )
    model = build(network)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    log(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
    history = fit(
        model,
        loader,
        epochs=EPOCHS,
        lr=LR,
        device=device,
        log=log,
        loss=LOSSES[loss],
        pretrained_lr=PRETRAINED_LR,
        ema=EMA.get(name),
    )
    return model, history


def run(name: str, folds_to_run: list[int] | None = None) -> dict:
    network, loss, label = EXPERIMENTS[name]
    t0 = time.time()

    def log(*args):
        print(f"[{time.time() - t0:5.0f}s]", *args, flush=True)

    device = pick_device()
    out = output_dir()
    root = find_data_root()
    train_dir, test_dir = root / "train" / "train_images", root / "test" / "test_images"
    annotations = load_annotations(next((root / "train").glob("*.json")))
    files = sorted(p.name for p in train_dir.iterdir())
    test_files = sorted(p.name for p in test_dir.iterdir())
    if QUICK:
        files, test_files = files[:20], test_files[:3]
    folds = fold_files(files, N_FOLDS, SEED)
    (out / f"{name}_folds.json").write_text(json.dumps({"seed": SEED, "folds": folds}, indent=1))
    folds_to_run = list(range(N_FOLDS)) if folds_to_run is None else folds_to_run
    log(f"{name}: {network}, loss {loss}, label {label}; folds {folds_to_run} of {[len(f) for f in folds]}; {device}")

    def frame(directory, f):
        return torch.from_numpy(channels(read_gray(directory / f)))

    report = {
        "name": name,
        "network": network,
        "loss": loss,
        "label": label,
        "quick": QUICK,
        "seed": SEED,
        "train_seed": TRAIN_SEEDS.get(name, SEED),
        "ema": EMA.get(name),
        "tile": TILE,
        "stride": STRIDE,
        "epochs": EPOCHS,
        "lr": LR,
        "pretrained_lr": PRETRAINED_LR,
        "quick_pq_params": A0,
        "folds": [],
    }
    for k in folds_to_run:
        longest = max((f["train_sec"] + f["predict_sec"] for f in report["folds"]), default=0)
        if time.time() - t0 + longest > BUDGET_SEC:
            log(f"stopping before fold {k}: it would not finish within the session")
            report["stopped_before_fold"] = k
            break
        start = time.time()
        held_out = folds[k]
        train_files = sorted(set(files) - set(held_out))
        log(f"fold {k}: {len(train_files)} frames")
        model, history = train(name, train_files, train_dir, annotations, device, TRAIN_SEEDS.get(name, SEED) + k, log)
        torch.save(model.state_dict(), out / f"{name}_fold{k}.pt")
        trained = time.time()

        oof = {f: predict_tiled(model, frame(train_dir, f), device) for f in held_out}
        np.savez_compressed(out / f"{name}_oof_fold{k}.npz", **oof)
        pq = PanopticQuality()
        for f, prob in oof.items():
            labels, areas, _ = instances(prob, **A0)
            for rles in annotations.rles_for_file(f):
                pq.update(instance_crops(rles), labels, areas)
        del oof
        test = {f: predict_tiled(model, frame(test_dir, f), device) for f in test_files}
        np.savez_compressed(out / f"{name}_test_fold{k}.npz", **test)
        del test, model
        report["folds"].append(
            {
                "fold": k,
                "train_frames": len(train_files),
                "held_out_frames": len(held_out),
                "loss": [round(v, 4) for v in history],
                "quick_pq": {**pq.summary(), "iou_sum": round(pq.iou_sum, 4)},
                "train_sec": round(trained - start, 1),
                "predict_sec": round(time.time() - trained, 1),
                "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if device.type == "cuda" else None,
            }
        )
        if device.type == "cuda":
            torch.cuda.empty_cache()
        log(f"fold {k} done:", report["folds"][-1]["quick_pq"])
        (out / f"{name}_report.json").write_text(json.dumps(report, indent=2))

    pooled = PanopticQuality()
    for fold in report["folds"]:
        s = fold["quick_pq"]
        pooled.tp, pooled.fp, pooled.fn = pooled.tp + s["tp"], pooled.fp + s["fp"], pooled.fn + s["fn"]
        pooled.iou_sum += s["iou_sum"]
    report["pooled_quick_pq"] = pooled.summary()
    report["elapsed_sec"] = round(time.time() - t0, 1)
    (out / f"{name}_report.json").write_text(json.dumps(report, indent=2))
    log("pooled quick PQ (A0 instances, no filter):", report["pooled_quick_pq"])
    return report


def run_full(name: str) -> dict:
    """The experiment trained once on all training frames, for the submission: `<name>_full.pt`,
    `<name>_test_full.npz` (its test maps, single model, made as the out-of-fold maps are), and
    `<name>_full_report.json`. Same schedule and epochs as each fold, so 25 % more steps."""
    t0 = time.time()

    def log(*args):
        print(f"[{time.time() - t0:5.0f}s]", *args, flush=True)

    device = pick_device()
    out = output_dir()
    root = find_data_root()
    train_dir, test_dir = root / "train" / "train_images", root / "test" / "test_images"
    annotations = load_annotations(next((root / "train").glob("*.json")))
    files = sorted(p.name for p in train_dir.iterdir())
    test_files = sorted(p.name for p in test_dir.iterdir())
    if QUICK:
        files, test_files = files[:20], test_files[:3]
    log(f"{name}: {EXPERIMENTS[name]} on all {len(files)} frames; {device}")
    model, history = train(name, files, train_dir, annotations, device, TRAIN_SEEDS.get(name, SEED) + N_FOLDS, log)
    torch.save(model.state_dict(), out / f"{name}_full.pt")
    trained = time.time()
    test = {f: predict_tiled(model, torch.from_numpy(channels(read_gray(test_dir / f))), device) for f in test_files}
    np.savez_compressed(out / f"{name}_test_full.npz", **test)
    report = {
        "name": name,
        "experiment": EXPERIMENTS[name],
        "quick": QUICK,
        "train_frames": len(files),
        "test_frames": len(test_files),
        "loss": [round(v, 4) for v in history],
        "train_sec": round(trained - t0, 1),
        "predict_sec": round(time.time() - trained, 1),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2) if device.type == "cuda" else None,
    }
    (out / f"{name}_full_report.json").write_text(json.dumps(report, indent=2))
    log(report)
    return report

In [ ]:
import subprocess
import sys

# Pin the model implementations across local checks and the Kaggle runtime.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "segmentation-models-pytorch==0.5.0", "timm==1.0.29"],
    check=True,
)

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from cnn_baseline.data import channels
from cnn_baseline.training import pick_device
from dataset.paths import find_data_root, find_input, output_dir
from models.swin_unet import SwinUNet
from postprocess.filtering import FEATURES, ComponentFilter, frame_context
from postprocess.pipeline import Postprocess
from preprocess.images import read_gray
from scoring import rle, submission
from scoring.check import check
from zoo.experiment import TILE, predict_views

MODEL = "swin_unet_boundary_full_fp16.pt"
CONFIG = "final_config.json"
FILTER_TABLE = "final_filter_table.csv"
SUBMITTED = "submission_tta8_h60_cut0.6.csv"  # the file sent to the leaderboard, to compare with
T0 = time.time()


def log(*args):
    print(f"[{time.time() - T0:5.0f}s]", *args, flush=True)


def stored(name: str) -> Path:
    """`weights/<name>` in the repository, or the same file attached to the Kaggle notebook."""
    local = Path("weights") / name
    return local if local.exists() else find_input(name)


device = pick_device()
model = SwinUNet(pretrained=False, tile=TILE)
model.load_state_dict({k: v.float() for k, v in torch.load(stored(MODEL), map_location="cpu").items()})
model.to(device).eval()
config = json.loads(stored(CONFIG).read_text())
table = pd.read_csv(stored(FILTER_TABLE))
post = Postprocess(
    ComponentFilter(cutoff=config["filter_cutoff"]).fit(table[FEATURES], table["target"].to_numpy()),
    params=config["params"],
    join=config["join"],
    mask_cut=config["mask_cut"],
)
log(f"model and post-processing loaded: {post.params}, cutoff {post.filter.cutoff}, mask cut {post.mask_cut}")

## Every test frame

In [ ]:
test_dir = find_data_root() / "test" / "test_images"
test_files = sorted(p.name for p in test_dir.iterdir())
rows = []
for i, f in enumerate(test_files, start=1):
    image = read_gray(test_dir / f)
    prob = predict_views(model, torch.from_numpy(channels(image)), device)
    labels, _, boxes = post(frame_context(f, image, prob))
    rows += submission.rows_from_label_map(f.rsplit(".", 1)[0], labels, boxes)
    if i % 30 == 0:
        log(f"{i}/{len(test_files)} frames, {len(rows)} filaments")
path = output_dir() / "submission.csv"
submission.write(rows, path)
check(path)

## Does it reproduce the submitted file?

Each submitted filament is paired with the new filament of the same frame that overlaps it most.

In [ ]:
try:
    submitted = pd.read_csv(find_input(SUBMITTED))
except FileNotFoundError:
    submitted = None
    log("submitted file not attached; skipping the comparison")
if submitted is not None:
    new = pd.DataFrame(rows)
    ious = []
    for frame_id, group in submitted.groupby(submitted["filament_id"].str.rsplit("_", n=1).str[0]):
        mine = [
            rle.decode(c).astype(bool)
            for c in new.loc[new["filament_id"].str.startswith(frame_id + "_"), "segmentation_rle"]
        ]
        for counts in group["segmentation_rle"]:
            mask = rle.decode(counts).astype(bool)
            best = max(((mask & m).sum() / (mask | m).sum() for m in mine), default=0.0)
            ious.append(best)
    ious = np.array(ious)
    log(
        f"submitted {len(submitted)} filaments, new {len(new)}; "
        f"identical {np.mean(ious == 1):.3f}, IoU > 0.95 {np.mean(ious > 0.95):.3f}, median IoU {np.median(ious):.4f}"
    )